# 7 · Enhancer redundancy — Step 2 (3D recompute) + analysis (local)

Notebook 6 (Spark) wrote the nearest-active-enhancer table per cell line + the ensemble map to S3. This local notebook (env `enhancer3D`, numpy available) does the heavy **Step 2** — recompute each gene's c1-nearest enhancer's **3D distance in c2's model** by loading c2 ensembles from `model-repository` — then builds the redundancy table, computes **frequency** per comparison, and compares **|log2FC|** between redundancy genes and non-redundant switchers.

Hypothesis: redundancy genes (old enhancer drifts away, a different one stays close) have smaller |log2FC| — conserved expression.

In [ ]:
import os.path, json
import numpy as np
import pandas as pd
import fsspec
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# --- inputs from notebook 6 (pull the two parquet dirs from S3 to here) ---
NEAREST_PATH = "../../data/whole_chromosomes/enhancer_redundancy_nearest"
ENSMAP_PATH  = "../../data/whole_chromosomes/enhancer_redundancy_ensmap"
DESEQ_DIR    = "../../data/deseq"
FIGS = "figs"; EXPORT = "export"

# 3D models: fsspec handles both S3 and a local pull. Point MODEL_REPOSITORY at
# "s3://model-repository" (set MODEL_STORAGE_OPTIONS with key/secret/endpoint) or a
# local directory you synced the ensembles into, e.g. "../../data/model-repository".
MODEL_REPOSITORY = "s3://model-repository"
MODEL_STORAGE_OPTIONS = {}   # e.g. {"key":"...","secret":"...","client_kwargs":{"endpoint_url":"https://..."}}

COMPARISONS = [("GM12878","H1ESC"), ("H1ESC","GM12878"),
               ("H1ESC","HFFC6"),   ("HFFC6","H1ESC"),
               ("GM12878","HFFC6"), ("HFFC6","GM12878")]

# comparison -> (deseq parquet, flip_log2fc?) — align sign to c1->c2 direction.
DESEQ = {
    "gm12878_vs_h1esc": ("gm12878_vs_h1esc_results.parquet", False),
    "h1esc_vs_gm12878": ("gm12878_vs_h1esc_results.parquet", True),
    "h1esc_vs_hffc6":   ("h1esc_vs_hffc6_results.parquet", False),
    "hffc6_vs_h1esc":   ("h1esc_vs_hffc6_results.parquet", True),
    "gm12878_vs_hffc6": ("gm12878_vs_hffc6_results.parquet", False),
    "hffc6_vs_gm12878": ("gm12878_vs_hffc6_results.parquet", True),
}

In [ ]:
# --- helpers ----------------------------------------------------------------

def chrom_tertile_thresholds(df, dist_col, chrom_col):
    # per-chromosome (q33, q67) of dist_col (matches notebook 3 tertiles)
    out = {}
    for chrom, grp in df.groupby(chrom_col):
        out[chrom] = (float(grp[dist_col].quantile(0.33)),
                      float(grp[dist_col].quantile(0.67)))
    return out

def proximity_categories(dist_series, chrom_series, thresholds):
    # small/mid/large vs per-chromosome thresholds; NaN or unknown chrom -> large
    cats = []
    for val, ch in zip(dist_series, chrom_series):
        t = thresholds.get(ch)
        if t is None or pd.isna(val):
            cats.append('large')
        elif val <= t[0]:
            cats.append('small')
        elif val <= t[1]:
            cats.append('mid')
        else:
            cats.append('large')
    return cats

def mean_distance_in_model(coords, gene_bins, enh_bins):
    # ensemble-mean Euclidean 3D distance for paired bins (mirrors models.py:80-83)
    gene_bins = np.asarray(gene_bins, dtype=int)
    enh_bins = np.asarray(enh_bins, dtype=int)
    a = coords[:, gene_bins, :]
    b = coords[:, enh_bins, :]
    return np.linalg.norm(a - b, axis=2).mean(axis=0)

_ENS_CACHE = {}
def load_ensemble(ensemble_id):
    # -> (coords (n_models, n_bins, 3), first_bin, last_bin, resolution)
    # mirrors load_chromatin_model_ensemble_from_filesystem (packed.py:9), via fsspec
    if ensemble_id in _ENS_CACHE:
        return _ENS_CACHE[ensemble_id]
    base = f"{MODEL_REPOSITORY}/{ensemble_id}"
    with fsspec.open(f"{base}.metadata.json", "r", **MODEL_STORAGE_OPTIONS) as fh:
        meta = json.load(fh)
    with fsspec.open(f"{base}.coordinates.npy", "rb", **MODEL_STORAGE_OPTIONS) as fh:
        coords = np.load(fh)
    out = (coords, int(meta['first_bin']), int(meta['last_bin']), int(meta['resolution']))
    _ENS_CACHE[ensemble_id] = out
    return out

def dist_L1_in_c2(pair_df, c2, ens_by):
    # 3D distance of each gene's c1-nearest enhancer (L1) measured in c2's model
    dists = np.full(len(pair_df), np.nan)
    oow = np.zeros(len(pair_df), dtype=bool)
    for chrom, idx in pair_df.groupby('gene_chr').groups.items():
        rows = pair_df.loc[idx]
        positions = pair_df.index.get_indexer(idx)
        ens_id = ens_by.get((c2, str(chrom)))
        if ens_id is None:
            oow[positions] = True
            continue
        coords, first_bin, last_bin, res = load_ensemble(ens_id)
        gene_tss = np.where(rows['gene_strand'].values == '+',
                            rows['gene_start_c1'].values, rows['gene_end_c1'].values)
        enh_ctr = (rows['enh_start_c1'].values + rows['enh_end_c1'].values) // 2
        in_win = ((gene_tss >= first_bin) & (gene_tss <= last_bin)
                  & (enh_ctr >= first_bin) & (enh_ctr <= last_bin))
        gbin = (gene_tss - first_bin) // res + 1
        ebin = (enh_ctr - first_bin) // res + 1
        if in_win.any():
            dists[positions[in_win]] = mean_distance_in_model(coords, gbin[in_win], ebin[in_win])
        oow[positions[~in_win]] = True
    return dists, oow

In [ ]:
# --- load Step-1 outputs + build the redundancy table per comparison (Step 2) ---
nearest_pd = pd.read_parquet(NEAREST_PATH)
nearest_pd['gene_chr'] = nearest_pd['gene_chr'].astype(str)
ensmap = pd.read_parquet(ENSMAP_PATH)
ENS_BY = {(r.cell_line, str(r.chrom)): r.ensemble_id for r in ensmap.itertuples()}
print("nearest:", nearest_pd.shape, "| ensembles mapped:", len(ENS_BY))

def build_comparison(c1, c2):
    n1 = nearest_pd[nearest_pd.cell_line == c1].set_index('gene_id')
    n2 = nearest_pd[nearest_pd.cell_line == c2].set_index('gene_id')
    th_c1 = chrom_tertile_thresholds(n1.rename(columns={'avg_dist': 'd'}), 'd', 'gene_chr')
    th_c2 = chrom_tertile_thresholds(n2.rename(columns={'avg_dist': 'd'}), 'd', 'gene_chr')

    genes = n1.index.intersection(n2.index)
    df = pd.DataFrame({'gene_id': list(genes)})
    df['gene_chr'] = n1.loc[genes, 'gene_chr'].values
    df['gene_start_c1'] = n1.loc[genes, 'gene_start'].values
    df['gene_end_c1'] = n1.loc[genes, 'gene_end'].values
    df['gene_strand'] = n1.loc[genes, 'gene_strand'].values
    df['nearest_enh_c1'] = n1.loc[genes, 'enh_id'].values
    df['enh_start_c1'] = n1.loc[genes, 'enh_start'].values
    df['enh_end_c1'] = n1.loc[genes, 'enh_end'].values
    df['avg_dist_c1'] = n1.loc[genes, 'avg_dist'].values
    df['nearest_enh_c2'] = n2.loc[genes, 'enh_id'].values
    df['avg_dist_c2'] = n2.loc[genes, 'avg_dist'].values

    df['proximity_category_c1'] = proximity_categories(df['avg_dist_c1'], df['gene_chr'], th_c1)
    df['proximity_category_c2'] = proximity_categories(df['avg_dist_c2'], df['gene_chr'], th_c2)

    # Step 2: c1-nearest enhancer distance recomputed in c2's model (out-of-window/NaN -> large)
    d, oow = dist_L1_in_c2(df, c2, ENS_BY)
    df['dist_L1_in_c2'] = d
    df['L1_out_of_c2_window'] = oow
    df['proximity_of_L1_in_c2'] = proximity_categories(df['dist_L1_in_c2'], df['gene_chr'], th_c2)

    df['nearest_changed'] = df['nearest_enh_c1'] != df['nearest_enh_c2']
    old_far = df['proximity_of_L1_in_c2'].eq('large')
    new_close = df['proximity_category_c2'].eq('small')
    df['is_redundancy'] = df['nearest_changed'] & old_far & new_close
    df['is_nonredundant_switcher'] = df['nearest_changed'] & old_far & ~new_close

    df.insert(0, 'comparison', f"{c1.lower()}_vs_{c2.lower()}")
    df.insert(1, 'c1', c1); df.insert(2, 'c2', c2)
    return df

tables = {}
for c1, c2 in COMPARISONS:
    t = build_comparison(c1, c2)
    key = f"{c1.lower()}_vs_{c2.lower()}"
    tables[key] = t
    t.to_parquet(os.path.join(EXPORT, f"redundancy_table_{key}.parquet"), index=False)
    print(f"{c1}->{c2}: genes={len(t)}  redundancy={int(t.is_redundancy.sum())}  "
          f"out_of_window={int(t.L1_out_of_c2_window.sum())}")

In [ ]:
# --- join DESeq2, frequency per comparison ---
def redundancy_frequency(df):
    denom = int(df['nearest_changed'].sum())
    return float('nan') if denom == 0 else int(df['is_redundancy'].sum()) / denom

def load_deseq(comparison):
    fname, flip = DESEQ[comparison]
    de = pd.read_parquet(os.path.join(DESEQ_DIR, fname)).reset_index()
    gid = next((c for c in ['gene_id', 'Geneid', 'index'] if c in de.columns), de.columns[0])
    de = de.rename(columns={gid: 'gene_id'})[['gene_id', 'log2FoldChange', 'padj']].copy()
    if flip:
        de['log2FoldChange'] = -de['log2FoldChange']
    de['gene_id'] = de['gene_id'].astype(str).str.split('.').str[0]
    return de

rows = []; joined = {}
for comparison, t in tables.items():
    de = load_deseq(comparison)
    t = t.copy(); t['gene_id'] = t['gene_id'].astype(str).str.split('.').str[0]
    m = t.merge(de, on='gene_id', how='inner')
    m['abs_log2fc'] = m['log2FoldChange'].abs()
    joined[comparison] = m
    rows.append({'comparison': comparison, 'n_genes': len(m),
                 'n_changed': int(m.nearest_changed.sum()),
                 'n_redundancy': int(m.is_redundancy.sum()),
                 'frequency': redundancy_frequency(m)})
freq_df = pd.DataFrame(rows)
freq_df.to_csv(os.path.join(EXPORT, "enhancer_redundancy_frequency.csv"), index=False)

ax = freq_df.plot.bar(x='comparison', y='frequency', legend=False, figsize=(8, 4))
ax.set_ylabel("redundancy frequency"); plt.tight_layout()
plt.savefig(os.path.join(FIGS, "exp6_frequency.png"), dpi=150); plt.show()
freq_df

In [ ]:
# --- |log2FC|: redundancy vs non-redundant switchers ---
def compare_abs_log2fc(redundancy_lfc, contrast_lfc):
    a = np.abs(np.asarray(redundancy_lfc, dtype=float)); a = a[~np.isnan(a)]
    b = np.abs(np.asarray(contrast_lfc, dtype=float)); b = b[~np.isnan(b)]
    n1, n2 = len(a), len(b)
    if n1 == 0 or n2 == 0:
        return {'n_redundancy': n1, 'n_contrast': n2, 'median_redundancy': np.nan,
                'median_contrast': np.nan, 'U': np.nan, 'pvalue': np.nan, 'rank_biserial': np.nan}
    res = stats.mannwhitneyu(a, b, alternative='two-sided')
    return {'n_redundancy': n1, 'n_contrast': n2,
            'median_redundancy': float(np.median(a)), 'median_contrast': float(np.median(b)),
            'U': float(res.statistic), 'pvalue': float(res.pvalue),
            'rank_biserial': 1.0 - (2.0 * res.statistic) / (n1 * n2)}

stat_rows = []
for comparison, m in joined.items():
    s = compare_abs_log2fc(m.loc[m.is_redundancy, 'abs_log2fc'].values,
                           m.loc[m.is_nonredundant_switcher, 'abs_log2fc'].values)
    s['comparison'] = comparison
    stat_rows.append(s)
stats_df = pd.DataFrame(stat_rows)[['comparison', 'n_redundancy', 'n_contrast', 'median_redundancy',
                                    'median_contrast', 'U', 'pvalue', 'rank_biserial']]
stats_df.to_csv(os.path.join(EXPORT, "enhancer_redundancy_log2fc_stats.csv"), index=False)
stats_df

In [ ]:
# --- distribution figures (ECDF + violin) ---
for comparison, m in joined.items():
    sub = m.assign(group=np.where(m.is_redundancy, 'redundancy',
                          np.where(m.is_nonredundant_switcher, 'non_redundant', None)))
    sub = sub.dropna(subset=['group'])
    if sub.empty:
        continue
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 4))
    for g, gg in sub.groupby('group'):
        xs = np.sort(gg['abs_log2fc'].values)
        a1.plot(xs, np.linspace(0, 1, len(xs)), label=g)
    a1.set_xlabel("|log2FC|"); a1.set_ylabel("ECDF"); a1.legend(); a1.set_title(comparison)
    sns.violinplot(data=sub, x='group', y='abs_log2fc', ax=a2, cut=0)
    a2.set_title("|log2FC| by group")
    plt.tight_layout()
    plt.savefig(os.path.join(FIGS, f"exp6_log2fc_{comparison}.png"), dpi=150)
    plt.show()